# PROJECT 9
by Szymon Waliczek
 - Ballistic **Andreev** transport in **2DEG** side **Josephson junction** - semi/super-conducting hybrid **InAs-SC**.
 - System is a wire with **2 semiconducting leads(up/down)** and **1 superconducting lead/block** on the right edge.
 - We analyze the relation of dispersion using wrapped system with Y translational symmetry.
 - **With Peierls phase**.
 - **No spin**.
 - With **type s** or **type d** electron pairing.

In [1]:
import ipyparallel as ipp
#-------------------------------------------------------------------------------------------------------------------
cluster = ipp.Client(profile="kwant_parallel")
#-------------------------------------------------------------------------------------------------------------------
# from tera import TeraClient
# cluster = TeraClient(username="swaliczek", profile_name="slurm")
#-------------------------------------------------------------------------------------------------------------------
v = cluster[:]
lview = cluster.load_balanced_view()
len(v)

30

In [2]:
%%px --local

import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [3]:
import ipywidgets as widgets
from time import perf_counter
from IPython.display import display
from matplotlib import pyplot as plt
from ipywidgets import interactive, HBox, VBox, fixed
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
#-------------------------------------------------------------------------------------------------------------------
class Timer():
    def __init__(self, name):
        self.name = name
        self.engine_id = os.environ.get('IPY_ENGINE_ID', '0')
    
    def __enter__(self):
        self.start = perf_counter()
        return self

    def __exit__(self, *args):
        self.end = perf_counter()
        if self.engine_id == '0':
            t = self.end - self.start
            print(f"Time - {self.name}: {t/60:.3f} min")

In [4]:
%%px --local

import kwant
import tinyarray
import numpy as np
from functools import lru_cache
from scipy.sparse.linalg import eigsh
#-------------------------------------------------------------------------------------------------------------------
tau_0 = tinyarray.array([[1, 0], [0, 1]])
tau_x = tinyarray.array([[0, 1], [1, 0]])
tau_y = tinyarray.array([[0, -1j], [1j, 0]])
tau_z = tinyarray.array([[1, 0], [0, -1]])
#-------------------------------------------------------------------------------------------------------------------
# Physical constants
PI = np.pi
from scipy.constants import physical_constants
eV = physical_constants['electron volt'][0]
m_eff = 0.023 * physical_constants['electron mass'][0]
h_bar = physical_constants['Planck constant over 2 pi'][0]
phi_0 = physical_constants['elementary charge over h-bar'][0]

In [5]:
%%px --local

# Geometry
L200 = 1200
W100 = 200
L800 = 800
W500 = 500
L1200 = 1200
W1000 = 800
freedom_deg = 2; # e + holes

In [6]:
%%px --local

#-------------------------------------------------------------------------------------------------------------------
def at(_a):
    a = _a
    t = (h_bar**2 / (2.0 * m_eff * (_a*1e-9)**2)) / eV
    return a, t
#-------------------------------------------------------------------------------------------------------------------
def onsite(site, t, mu, B, delta, x0):
    return (4*t - mu) * tau_z
#-------------------------------------------------------------------------------------------------------------------
def onsite_s_wave(site, t, mu, B, delta, x0):
    return (4*t - mu)*tau_z + delta*tau_x
#-------------------------------------------------------------------------------------------------------------------
def hop_s_wave(site1, site2, t, mu, B, delta, x0):
    x1, y1 = site1.pos
    x2, y2 = site2.pos
    if x1 < 0:
        phi = phi_0 * B * ((x1 + x2)/2 + x0) * (y1 - y2)*1e-18
        p_phase = tinyarray.array([[np.exp(-1j * phi), 0], 
                              [0, np.exp(1j * phi)]])
        return -t * tau_z * p_phase
    return -t * tau_z * 1.0 # A=0
#-------------------------------------------------------------------------------------------------------------------
def hop_d_wave(site1, site2, t, mu, B, delta, x0):
    x1, y1 = site1.pos
    x2, y2 = site2.pos
    d_wave = -(delta*tau_x)
    if (x1 != x2): d_wave = (delta*tau_x)
    
    if x1 < 0:
        phi = phi_0 * B * ((x1 + x2)/2 + x0) * (y1 - y2)*1e-18
        p_phase = tinyarray.array([[np.exp(-1j * phi), 0], 
                              [0, np.exp(1j * phi)]])
        return -t*tau_z*p_phase + 0.0 # d=0
    else:
        return -t*tau_z*1.0 + d_wave # A=0
#-------------------------------------------------------------------------------------------------------------------    
def make_system(a, W, Wsc, L, f, type, hybrid, wrap):
    lat = kwant.lattice.square(a, norbs=f)
    # SC hybrid
    if(hybrid==True):
        sys_hybrid = kwant.Builder()
        sys_hybrid[lat.shape(lambda pos: -W <= pos[0] < 0 and -L/2 <= pos[1] <= L/2, (-a, 0))] = onsite # Site N
        if (type=='s'): sys_hybrid[lat.shape(lambda pos: pos[0] == 0 and -L/2 <= pos[1] <= L/2, (0, 0))] = onsite_s_wave # Site SC
        if (type=='d'): sys_hybrid[lat.shape(lambda pos: pos[0] == 0 and -L/2 <= pos[1] <= L/2, (0, 0))] = onsite # Site SC
        if (type=='s'): sys_hybrid[lat.neighbors()] = hop_s_wave
        if (type=='d'): sys_hybrid[lat.neighbors()] = hop_d_wave
        lead_n = kwant.Builder(kwant.TranslationalSymmetry((0, -a)), conservation_law=-tau_z, particle_hole=tau_y) # Lead N
        lead_n[lat.shape(lambda pos: -W <= pos[0] < 0, (-a, 0))] = onsite
        if (type=='s'): lead_n[lat.neighbors()] = hop_s_wave
        if (type=='d'): lead_n[lat.neighbors()] = hop_d_wave
        sys_hybrid.attach_lead(lead_n);
        sys_hybrid.attach_lead(lead_n.reversed());
        lead_sc = kwant.Builder(kwant.TranslationalSymmetry((a, 0))) # Lead SC
        if (type=='s'): lead_sc[lat.shape(lambda pos: -L/2 <= pos[1] <= L/2, (0, 0))] = onsite_s_wave
        if (type=='d'): lead_sc[lat.shape(lambda pos: -L/2 <= pos[1] <= L/2, (0, 0))] = onsite
        if (type=='s'): lead_sc[lat.neighbors()] = hop_s_wave    
        if (type=='d'): lead_sc[lat.neighbors()] = hop_d_wave    
        sys_hybrid.attach_lead(lead_sc)
        sys_hybrid=sys_hybrid.finalized()
#-------------------------------------------------------------------------------------------------------------------
    block=False # SC block
    if(block==True):
        sys_block = kwant.Builder()
        sys_block[lat.shape(lambda pos: -W <= pos[0] < 0 and -L/2 <= pos[1] <= L/2, (-a, 0))] = onsite # Site N
        if (Wsc>0 and type=='s'): sys_block[lat.shape(lambda pos: 0 <= pos[0] <= W_sc_block and -L/2 <= pos[1] <= L/2, (0, 0))] = onsite_s_wave # Site SC
        if (Wsc>0 and type=='d'): sys_block[lat.shape(lambda pos: 0 <= pos[0] <= W_sc_block and -L/2 <= pos[1] <= L/2, (0, 0))] = onsite # Site SC
        if (type=='s'):sys_block[lat.neighbors()] = hop_s_wave
        if (type=='d'):sys_block[lat.neighbors()] = hop_d_wave
        sys_block.attach_lead(lead_n)
        sys_block.attach_lead(lead_n.reversed())
        sys_block=sys_block.finalized()
#-------------------------------------------------------------------------------------------------------------------
     # Wraparound
    if(wrap==True):
        sys_wrap = kwant.Builder(kwant.TranslationalSymmetry((0, a)))
        if (W>0): sys_wrap[lat.shape(lambda pos: -W <= pos[0] < 0, (-a, 0))] = onsite
        if (Wsc>0 and type=='s'): sys_wrap[lat.shape(lambda pos: 0 <= pos[0] <= Wsc, (0, 0))] = onsite_s_wave
        if (Wsc>0 and type=='d'): sys_wrap[lat.shape(lambda pos: 0 <= pos[0] <= Wsc, (0, 0))] = onsite
        if (type=='s'): sys_wrap[lat.neighbors()] = hop_s_wave
        if (type=='d'): sys_wrap[lat.neighbors()] = hop_d_wave
        sys_wrap=kwant.wraparound.wraparound(sys_wrap, coordinate_names='y').finalized()
    if(hybrid==True and wrap==True): return sys_hybrid, sys_wrap
    if(hybrid==True): return sys_hybrid
    if(wrap==True): return sys_wrap
#-------------------------------------------------------------------------------------------------------------------
@lru_cache(maxsize=1)
def initialize_wrap(_a, _W, _Wsc, _L, _f, _type):
    ini_wrap = make_system(a=_a, W=_W, Wsc=_Wsc, L=_L, f=_f, type=_type, hybrid=False, wrap=True)
    return ini_wrap
#-------------------------------------------------------------------------------------------------------------------
def compute_eigen(sys_wrap, ky, _modes, _t, _mu, _B, _delta, _x0):
    p = dict(t=_t, mu=_mu, B=_B, delta=_delta, x0=_x0, k_y=ky)
    H = sys_wrap.hamiltonian_submatrix(params=p, sparse=True)
    E, V = eigsh(H, k=_modes, sigma=0, which='LM')
    idx = E.real.argsort()
    return E[idx].real, V[:, idx]
#-------------------------------------------------------------------------------------------------------------------
def compute_modes_lead(lead, _E, _t, _mu, _B, _delta, _x0):
    p = dict(t=_t, mu=_mu, B=_B, delta=_delta, x0=_x0)
    prop_modes, _ = lead.modes(energy=_E, params=p)
    return len(prop_modes.momenta)
#-------------------------------------------------------------------------------------------------------------------
@lru_cache(maxsize=1)
def initialize_lead(a, W, Wsc, L, f, type):
    lat = kwant.lattice.square(a, norbs=f) 
    lead_hybrid = kwant.Builder(kwant.TranslationalSymmetry((0, -a)), particle_hole=tau_y)
    if (W>0): lead_hybrid[lat.shape(lambda pos: -W <= pos[0] < 0, (-a, 0))] = onsite
    if (Wsc>0 and type=='s'): lead_hybrid[lat.shape(lambda pos: 0 <= pos[0] < Wsc, (0, 0))] = onsite_s_wave
    if (Wsc>0 and type=='d'): lead_hybrid[lat.shape(lambda pos: 0 <= pos[0] < Wsc, (0, 0))] = onsite
    if (type=='s'): lead_hybrid[lat.neighbors()] = hop_s_wave
    if (type=='d'): lead_hybrid[lat.neighbors()] = hop_d_wave
    return lead_hybrid.finalized()
#-------------------------------------------------------------------------------------------------------------------
def get_dk_lead(_a, _W, _Wsc, _L, _f, _type, _dk_mode, _E, _t, _mu, _B, _delta, _x0):
    p = dict(t=_t, mu=_mu, B=_B, delta=_delta, x0=_x0)
    prop_modes, _ = initialize_lead(a=_a, W=_W, Wsc=_Wsc, L=_L, f=_f, type=_type).modes(energy=_E, params=p)
    momentum = prop_modes.momenta
    k_vals = np.sort(momentum[momentum > 0])
    if len(k_vals) == 0: return 0.0
    return 2.0 * k_vals[_dk_mode]

In [7]:
def print_params(a, t):
    print("Parameters:")
    print(f"h_bar = {h_bar}")
    print(f"m_eff = {m_eff}")
    print(f"phi_0 = {phi_0}")
    print(f"a = {a} nm")
    print(f"t = {t} eV")
#-------------------------------------------------------------------------------------------------------------------
def plot_sys(sys, a, x, y, name, p):
    with Timer(name):
        COLOR = lambda site: np.real(sys.hamiltonian(site, site, params = p)[0,1])
        kwant.plot(sys, fig_size=(x, y), show=False, site_color = COLOR)
        plt.title(f"{name} a = {a}nm")
        plt.xlabel("x [nm]")
        plt.ylabel("y [nm]")
        plt.show()
#-------------------------------------------------------------------------------------------------------------------
def eigen(a, W, Wsc, L, f, type, modes, xlim, nk, p):
    with Timer('eigen'):
        k_range = np.linspace(-xlim, xlim, nk)
        results = lview.map_sync(lambda k: compute_eigen(initialize_wrap(a, W, Wsc, L, f, type), k, modes,
                                                         p['t'], p['mu'], p['B'], p['delta'], p['x0']), k_range)
        E = np.array([x[0] for x in results])
        V = np.array([x[1] for x in results])
    return E, k_range, V
#-------------------------------------------------------------------------------------------------------------------
def eigen_ana(a, W, xlim, nk, n, p):
    with Timer('eigen_ana'):
        E = np.zeros((n, nk))
        ky = np.linspace(-xlim, xlim, nk)
        for i in range(n):
            kx = (i + 1) * PI / (W+a)
            
            for j in range(nk): 
                k_sq = ((kx*1e9)**2 + (ky[j]*1e9/a)**2)
                E[i][j] = (h_bar**2 * k_sq) / (2.0 * m_eff) / eV - p['mu']
    return E
#-------------------------------------------------------------------------------------------------------------------
def compare_numerical_analytical(a, W, Wsc, L, name, En, Ea, k_range, dk, s, xlim, ylim, n, p):
    modes = En.shape[1]
    plt.figure(figsize=(s,s))
    for i in range(modes):
        plt.plot(k_range, En[:, i]*1e3, 'k.', markersize=1)
    #for i in range(n):
        #plt.plot(k_range, Ea[i][:]*1e3, '--', color='orchid', markersize=0.1)
    info = (f"{name} \na = {a} nm \nW = {W} nm \nWsc = {Wsc} nm \nL = {L} nm \n$\mu = {p['mu']*1e3:.0f}$ meV \n$B = {p['B']:.1f}$ T\
    \n$\Delta = {p['delta']*1e3}$ meV \n$\delta_k = {dk:.4f}$ 1/a")
    plt.text(1.1, 0.5, info, transform=plt.gca().transAxes, fontsize=12, verticalalignment='center')
    plt.hlines(y=0, xmin=-dk/2, xmax=dk/2, colors='red', label='$\delta_k$', lw=1)
    plt.axhline(p['delta']*1e3, color='blue', label='$\Delta$', lw=0.2)
    plt.axhline(p['mu']*1e3, color='seagreen', label='$\mu$', lw=0.3)
    plt.axhline(-p['delta']*1e3, color='blue', lw=0.2)
    plt.ylabel("E [meV]"); plt.yticks(np.arange(-ylim, ylim+ylim/10, ylim/50)); plt.ylim(-ylim, ylim)
    plt.xlabel("$k_y$ [1/a]"); plt.xlim(-xlim, xlim); plt.xticks(np.arange(-xlim, xlim+xlim/10, xlim/2))
    plt.legend(loc='upper right', fontsize=2*s); plt.grid(alpha=0.3); plt.show
#-------------------------------------------------------------------------------------------------------------------
def plot_bands_wrap(a, W, Wsc, L, name, E, k_range, dk, s, xlim, ylim, p):
    modes = E.shape[1]
    plt.figure(figsize=(s,s))
    for i in range(modes):
        plt.plot(k_range, E[:, i]*1e3, 'k.', markersize=1)
    info = (f"{name} \na = {a} nm \nW = {W} nm \nWsc = {Wsc} nm \nL = {L} nm \n$\mu = {p['mu']*1e3:.0f}$ meV \n$B = {p['B']:.1f}$ T\
    \n$\Delta = {p['delta']*1e3}$ meV \n$\delta_k = {dk:.4f}$ 1/a")
    plt.text(1.1, 0.5, info, transform=plt.gca().transAxes, fontsize=12, verticalalignment='center')
    plt.hlines(y=0, xmin=-dk/2, xmax=dk/2, colors='red', label='$\delta_k$', lw=1)
    plt.axhline(p['delta']*1e3, color='blue', label='$\Delta$', lw=0.2)
    plt.axhline(p['mu']*1e3, color='seagreen', label='$\mu$', lw=0.3)
    plt.axhline(-p['delta']*1e3, color='blue', lw=0.2)
    plt.ylabel("E [meV]"); plt.yticks(np.arange(-ylim, ylim+ylim/10, ylim/20)); plt.ylim(-ylim, ylim)
    plt.xlabel("$k_y$ [1/a]"); plt.xlim(-xlim, xlim); plt.xticks(np.arange(-xlim, xlim+xlim/10, xlim/2))
    plt.legend(loc='upper right', fontsize=2*s); plt.grid(alpha=0.3); plt.show
#-------------------------------------------------------------------------------------------------------------------
def interactive_E_k(_a, _W, _Wsc, _L, _name, _s, _f, type, _modes, _xlim, _ylim, _nk, _n, dk_mode, E0, 
                    _t, _mu, _B, _delta, _x0):
    _p = dict(t=_t, mu=_mu, B=_B, delta=_delta, x0=_x0)
    dk_lead = get_dk_lead(_a=_a, _W=_W, _Wsc=_Wsc, _L=_L, _f=_f, _type=type, _dk_mode=dk_mode, _E=E0, 
                          _t=_t, _mu=_mu, _B=_B, _delta=_delta, _x0=_x0)
    En_vals, k_vals, _ = eigen(a=_a, W=_W, Wsc=_Wsc, L=_L, f=_f, type=type, modes=_modes, xlim=_xlim, nk=_nk, p=_p)
    Ea_vals = eigen_ana(a=_a, W=(_W+_Wsc), xlim=_xlim, nk=_nk, n=_n, p=_p)
    compare_numerical_analytical(a=_a, W=_W, Wsc=_Wsc, L=_L, name=_name, En=En_vals, Ea=Ea_vals, k_range=k_vals, 
                                 dk=dk_lead, s=_s, xlim=_xlim, ylim=_ylim, n=_n, p=_p)
#-------------------------------------------------------------------------------------------------------------------
def analytical_xs_G(_sys, _a, _W, _WSC, _L, f, type, _alpha, _beta, p, dk_mode):
    with Timer('Analytical'):
        mu_vec = np.linspace(p['mu0'], p['mu1'], p['N'])
        def analytical(a, L, alpha, beta, dk): 
            return 1.0 - 8.0*(alpha*beta)**2 * np.sin(dk*L/a/2.0)**2
        results = lview.map_sync(lambda mu: get_dk_lead(_a=_a, _W=_W, _Wsc=_WSC, _L=_L, _f=f, _type=type, 
                                                        _t=p['t'], _mu=mu, _B=p['B'], _delta=p['delta'], _x0=p['x0'], 
                                                        _mode=dk_mode), mu_vec)
        dk_vec = np.array(results)
        g = lview.map_sync(lambda _dk: analytical(a=_a, L=_L, alpha=_alpha, beta=_beta, dk=_dk), dk_vec)
        return np.array(g)                                                  
#-------------------------------------------------------------------------------------------------------------------
def numerical_xs_G(_sys, p1):
    with Timer('Numerical'):
        mu_vec = np.linspace(p1['mu0'], p1['mu1'], p1['N'])
        p_base = [dict(t=['t'], mu=m, B=p1['B'], delta=p1['delta'], x0=p1['x0']) for m in mu_vec]
        g = lview.map_sync(lambda _p: compute_Gj(sys=_sys, E=p1['E'], p=_p, j=1), p_base)
        return np.array(g)
#-------------------------------------------------------------------------------------------------------------------
def plot_xs_G(Ga, Gn, name, L, s, ylim, p):
    plt.figure(figsize=(s, 4*s/5))
    mu_vec = np.linspace(p['mu0'], p['mu1'], p['N'])
    plt.plot(mu_vec*1e3, Ga, 'blue', label=f'Analytical P_e-P_h')
    plt.plot(mu_vec*1e3, Gn, 'red', label=f'Numerical G')
    info = (f"{name} \n$L = {L1200}$ nm \n$E = {p['E']*1e3:.0f}$ meV\n$\Delta = {p['delta']*1e3:.0f}$ meV \n$B={p['B']:.1}$ T")
    plt.text(1.1, 0.5, info, transform=plt.gca().transAxes, fontsize=12, verticalalignment='center')
    plt.ylabel('G / P [$e^2/h$]'); plt.yticks(np.arange(-ylim, ylim+ylim/10, 0.2)); plt.ylim(-ylim, ylim)
    plt.xlabel("$\mu$ [meV]"); plt.xticks(np.arange(p['mu0']*1e3, p['mu1']*1e3+1, 1))
    plt.legend(loc='upper right', fontsize='xx-small');    
    plt.title(f'G ( $\mu$ ) [ $e^2/h$ ]')
    plt.grid(True); plt.show()

In [8]:
%%px --local

a10, t10 = at(10)
a5, t5 = at(5)
a3, t3 = at(3)
a2, t2 = at(2)

In [9]:
Dispersion_relation = interactive(
    interactive_E_k,
    _a=fixed(a2), _W=fixed(0), _Wsc=fixed(200), _L=fixed(0), 
    _name=fixed('d-wave'), _s=fixed(14), _f=fixed(freedom_deg), type=fixed('d'), 
    _modes=fixed(10), _xlim=fixed(PI/20), _ylim=fixed(0.05), _nk=fixed(1000), _n=fixed(4), dk_mode=fixed(0), 
    E0=fixed(0), _t=fixed(t2),
    _mu=widgets.FloatSlider(min=0, max=0.01, step=0.001, value=0.004, description='mu:', readout_format='.3f'),
    _B=widgets.FloatSlider(min=-3, max=0, step=0.1, value=0, description='B:', readout_format='.1f'),
    _delta=widgets.FloatSlider(min=0, max=0.02, step=0.0005, value=0.02/(a2**2), description='delta:', readout_format='.4f'),
    _x0=widgets.FloatSlider(min=0, max=10, step=0.1, value=0, description='x0:', readout_format='.1f')
);
controls = VBox(Dispersion_relation.children[:-1])
output = Dispersion_relation.children[-1]
display(HBox([output, controls]))

In [198]:
Dispersion_relation = interactive(
    interactive_E_k,
    _a=fixed(a3), _W=fixed(0), _Wsc=fixed(200), _L=fixed(0), 
    _name=fixed('d-wave'), _s=fixed(14), _f=fixed(freedom_deg), type=fixed('d'), 
    _modes=fixed(10), _xlim=fixed(PI/20), _ylim=fixed(0.1), _nk=fixed(10000), _n=fixed(4), dk_mode=fixed(0), 
    E0=fixed(0), _t=fixed(t3),
    _mu=widgets.FloatSlider(min=0, max=0.01, step=0.001, value=0.004, description='mu:', readout_format='.3f'),
    _B=widgets.FloatSlider(min=-3, max=0, step=0.1, value=0, description='B:', readout_format='.1f'),
    _delta=widgets.FloatSlider(min=0, max=0.02, step=0.0005, value=0.02/(a2**2), description='delta:', readout_format='.4f'),
    _x0=widgets.FloatSlider(min=0, max=10, step=0.1, value=0, description='x0:', readout_format='.1f')
);
controls = VBox(Dispersion_relation.children[:-1])
output = Dispersion_relation.children[-1]
display(HBox([output, controls]))

In [69]:
p_test = dict(t=t5, mu = 0, B = 0, delta = 0.001, x0 = 0, k_y=0)

In [209]:
Dispersion_relation = interactive(
    interactive_E_k,
    _a=fixed(a5), _W=fixed(0), _Wsc=fixed(200), _L=fixed(0), 
    _name=fixed('d-wave'), _s=fixed(14), _f=fixed(freedom_deg), type=fixed('d'), 
    _modes=fixed(10), _xlim=fixed(PI/10), _ylim=fixed(0.3), _nk=fixed(10000), _n=fixed(4), dk_mode=fixed(0), 
    E0=fixed(0), _t=fixed(t5),
    _mu=widgets.FloatSlider(min=0, max=0.01, step=0.001, value=0.004, description='mu:', readout_format='.3f'),
    _B=widgets.FloatSlider(min=-3, max=0, step=0.1, value=0, description='B:', readout_format='.1f'),
    _delta=widgets.FloatSlider(min=0, max=0.02, step=0.0005, value=0.02/(a2**2), description='delta:', readout_format='.4f'),
    _x0=widgets.FloatSlider(min=0, max=10, step=0.1, value=0, description='x0:', readout_format='.1f')
);
controls = VBox(Dispersion_relation.children[:-1])
output = Dispersion_relation.children[-1]
display(HBox([output, controls]))

In [210]:
def calc(_a, _t, W, n, _mu, _B, _delta, _x0):
    _p = dict(t=_t, mu=_mu, B=_B, delta=_delta, x0=_x0)
    kx = np.zeros(n)
    ky = np.zeros(n)
    Delta_trig = np.zeros(n)
    Delta_quad = np.zeros(n)
    E = np.zeros(n)
    prop_modes, _ = initialize_lead(_a, W, 0, 0, f=freedom_deg, type='s').modes(energy=0, params=_p)
    ky = prop_modes.momenta[prop_modes.momenta > 0] # [1/a] to ahve [1/nm] must divide ky / a
    print(f"a= {_a} nm \tWsc = {W} nm \tDelta = {1e3*_delta} meV \nt = {_t*1e3:.3f} meV")
    for i in range(n):
        kx[i] = (i+1)*PI/(W200+_a) # 1/nm
        E[i] = h_bar**2*((kx[i]*1e9)**2 + (ky[n-1-i]/_a*1e9)**2) / (2.0 * m_eff) / eV - _mu
        # print(f"kx^2+ky^2 = {(kx[i]/1e-9)**2 + (ky[i]/(a*1e-9))**2}")
        Delta_trig[i] = abs(_p['delta']*(np.cos(_a*kx[i]) - np.cos(ky[n-1-i]))) # kx*a [1/nm * 10 nm] and ky is already dimensionless
        Delta_quad[i] = abs(_p['delta']*0.5*((ky[n-1-i])**2 - (kx[i]*_a)**2))
        #
        print(f"n = {i+1} \tE = {E[i]*1e3:.3f} meV \tDelta_trig = {1e6*Delta_trig[i]:.3f} [ueV] \tDelta_quad = {1e6*Delta_quad[i]:.3f} [ueV]  \tkx = {kx[i]:.3f} [1/nm] \tky = {ky[n-1-i]/_a:.3f} [1/nm]\n")
    print("=================================================================================================")
#calc(_a=a10, _t=t10, W=W200, n=3, _mu=0.004, _B=0, _delta=0.01, _x0=0)
calc(_a=a2, _t=t2, W=W200, n=3, _mu=0.004, _B=0, _delta=0.02, _x0=0)
calc(_a=a3, _t=t3, W=W200, n=3, _mu=0.004, _B=0, _delta=0.02, _x0=0)
calc(_a=a5, _t=t5, W=W200, n=3, _mu=0.004, _B=0, _delta=0.02, _x0=0)

a= 2 nm 	Wsc = 200 nm 	Delta = 20.0 meV 
t = 414.128 meV
n = 1 	E = 0.003 meV 	Delta_trig = 77.240 [ueV] 	Delta_quad = 77.302 [ueV]  	kx = 0.016 [1/nm] 	ky = 0.047 [1/nm]

n = 2 	E = 0.002 meV 	Delta_trig = 19.212 [ueV] 	Delta_quad = 19.228 [ueV]  	kx = 0.031 [1/nm] 	ky = 0.038 [1/nm]

n = 3 	E = 0.003 meV 	Delta_trig = 77.438 [ueV] 	Delta_quad = 77.500 [ueV]  	kx = 0.047 [1/nm] 	ky = 0.015 [1/nm]

a= 3 nm 	Wsc = 200 nm 	Delta = 20.0 meV 
t = 184.057 meV
n = 1 	E = -0.002 meV 	Delta_trig = 173.790 [ueV] 	Delta_quad = 174.105 [ueV]  	kx = 0.015 [1/nm] 	ky = 0.047 [1/nm]

n = 2 	E = -0.028 meV 	Delta_trig = 43.285 [ueV] 	Delta_quad = 43.363 [ueV]  	kx = 0.031 [1/nm] 	ky = 0.038 [1/nm]

n = 3 	E = -0.065 meV 	Delta_trig = 173.908 [ueV] 	Delta_quad = 174.219 [ueV]  	kx = 0.046 [1/nm] 	ky = 0.015 [1/nm]

a= 5 nm 	Wsc = 200 nm 	Delta = 20.0 meV 
t = 66.261 meV
n = 1 	E = 0.017 meV 	Delta_trig = 486.309 [ueV] 	Delta_quad = 488.774 [ueV]  	kx = 0.015 [1/nm] 	ky = 0.047 [1/nm]

n = 2 	E = 0.011

In [80]:
sh_l_1200, sw_1000 = make_system(a, W1000, W1000, L1200, t, f=freedom_deg, type='s', hybrid=True, wrap=True)
dh_l_1200, dw_1000 = make_system(a, W1000, W1000, L1200, t, f=freedom_deg, type='d', hybrid=True, wrap=True)

In [81]:
p0 = dict(E=0, mu0=0.003, mu1=0.006, B=-0.8, delta=0.001, x0=0, N=500)

In [85]:
Gana_sh_l_1200_0 = analytical_xs_G(_sys=sh_l_1200, _a =a, _W=W1000, _WSC=W1000, _L=L1200, _t=t, f=freedom_deg, type='s', _alpha=np.sqrt(0.5), _beta=np.sqrt(0.5), p=p0, dk_mode=0)
Gnum_sh_l_1200_0 = numerical_xs_G(_sys=sh_l_1200, p1=p0)

Time - Analytical: 0.548 min
Time - Numerical: 0.004 min


PicklingError: Can't pickle <function hop_s_wave at 0x7fbb9d212280>: it's not the same object as __main__.hop_s_wave

In [27]:
Gana_dh_l_1200_0 = analytical_xs_G(_sys=dh_l_1200, _a =a, _W=W1000, _WSC=W1000, _L=L1200, _t=t, f=freedom_deg, type='s', _alpha=np.sqrt(0.5), _beta=np.sqrt(0.5), p=p0)
Gnum_dh_l_1200_0 = numerical_xs_G(_sys=dh_l_1200, p1=p0)

Time - Analytical: 0.524 min
Time - Numerical: 0.337 min


In [ ]:
plot_xs_G(Ga=Gana_sh_l_1200_0, Gn=Gnum_sh_l_1200_0, name='s-wave-v2', L=L1200, s=4, ylim=1, p=p0)

In [ ]:
plot_xs_G(Ga=Gana_dh_l_1200_0, Gn=Gnum_dh_l_1200_0, name='d-wave-v2', L=L1200, s=4, ylim=1, p=p0)